# Your first scraper
In this project, we will guide you step by step through the process of:

1. creating a self-contained development environment.
1. retrieving some information from an API (a website for computers)
2. leveraging it to scrape a website that does not provide an API
3. saving the output for later processing

Here we query an API for a list of countries and their past leaders. We then extract and sanitize their short bio from Wikipedia. Finally, we save the data to disk.

This task is often the first (coding) step of a datascience project and you will often come back to it in the future.

You will study topics such as *scraping*, *data structures*, *regular expressions*, *concurrency* and *file handling*. We will point out useful resources at the appropriate time. 

Let's dive in!

## 0. Creating a clean environment

Use the [`venv`](https://docs.python.org/3/library/venv.html) command to create a new environment called `wikipedia_scraper_env`.

Activate it and add it to you `.gitignore` file. 

You will find more info about virtual environments in the course content and on the web.

## 1. API Scraping

### 1a. A simple API query
You will start with the basics: how to do a simple request to an [API endpoint](../../2.python/2.python_advanced/05.Scraping/5.apis.ipynb).

You will use the [requests](https://requests.readthedocs.io/en/latest/) external library through the `import` keyword. NOTE: external libraries need to be installed first. Check the [request Quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/) section of the documentation to:

1. Use the `get()` method to connect to this endpoint: https://country-leaders.onrender.com/status
2. Check if the `status_code` is equal to 200, which means OK.
    * if OK, `print()` the `text`` of the response.
    * if not, `print()` the `status_code`. 

Here is an explanation of [HTTP status codes](https://en.wikipedia.org/wiki/List_of_HTTP_status_codes).


In [4]:
import requests

# assign the root url (without /status) and the /status endpoint
root_url = "https://country-leaders.onrender.com"
status_url = root_url + "/status"

# query the /status endpoint using the get() method
req = requests.get(status_url)

# check the status_code and print appropriate messages
if req.status_code == 200:
    print(req.text)
else:
    print(req.status_code)

"Alive"


### 1b. Dealing with JSON

[JSON](https://quickref.me/json) is the preferred format to deal with data over the web. You cannot avoid it so you would better get acquainted.

Connect to another endpoint called `/countries` but this time the API will return data in the JSON format. 


In [5]:
# Set the countries_url variable
countries_url = root_url + "/countries"

# query the /countries endpoint using the get() method
req = requests.get(countries_url)

# Get the JSON content and store it in the countries variable
countries = req.json()

# display the request's status code and the countries variable
print(req.status_code, countries)

403 {'message': 'The cookie is missing'}


### 1c. Cookies anyone?

It looks like the access to this API is restricted...
Query the `/cookie` endpoint and extract the appropriate field to access your cookie.

You will need to use this cookie in each of the following API requests.

In [12]:
# Set the cookie_url variable
cookie_url = root_url + "/cookie"

# Query the endpoint, set the cookies variable and display it
cookies = requests.get(cookie_url).cookies
print(cookies)

<RequestsCookieJar[<Cookie user_cookie=ab05c9b0-ba5c-4cfa-81c2-cea0170675f3 for country-leaders.onrender.com/>]>


Try to query the countries endpoint using the cookie, save the output and print it.

In [13]:
# query the /countries endpoint using the cookie
countries = requests.get(countries_url, cookies=cookies).json()

# display the countries variable
print(countries)

['fr', 'us', 'be', 'ma', 'ru']


Chances are the cookie has expired... Thanksfully, you got a nice error message. For now, simply execute the last 2 cells quickly so you get a result.

### 1d. Getting the actual data from the API

Query the `/leaders` endpoint.

In [14]:
# Set the leaders_url variable
leaders_url = root_url + "/leaders"

# query the /leaders endpoint (without parameters yet)
leaders = requests.get(leaders_url, cookies=cookies).json()

# display the leaders variable
print(leaders)

{'message': 'Please specify a country'}


It looks like this endpoint requires additional information in order to return its result. Check the API [*documentation*](https://country-leaders.onrender.com/docs) in your web browser.

Change the query to accept *parameters*. You should know where to find help by now.

In [15]:
# query the /leaders endpoint using cookies and parameters (take any country in countries)
leaders = requests.get(leaders_url, cookies=cookies, params={"country": "be"}).json()

# display the leaders variable
print(leaders)

[{'id': 'Q12978', 'first_name': 'Guy', 'last_name': 'Verhofstadt', 'birth_date': '1953-04-11', 'death_date': None, 'place_of_birth': 'Dendermonde', 'wikipedia_url': 'https://nl.wikipedia.org/wiki/Guy_Verhofstadt', 'start_mandate': '1999-07-12', 'end_mandate': '2008-03-20'}, {'id': 'Q12981', 'first_name': 'Yves', 'last_name': 'Leterme', 'birth_date': '1960-10-06', 'death_date': None, 'place_of_birth': 'Wervik', 'wikipedia_url': 'https://nl.wikipedia.org/wiki/Yves_Leterme', 'start_mandate': '2009-11-25', 'end_mandate': '2011-12-06'}, {'id': 'Q12983', 'first_name': 'Herman', 'last_name': 'None', 'birth_date': '1947-10-31', 'death_date': None, 'place_of_birth': 'Etterbeek', 'wikipedia_url': 'https://nl.wikipedia.org/wiki/Herman_Van_Rompuy', 'start_mandate': '2008-12-30', 'end_mandate': '2009-11-25'}, {'id': 'Q14989', 'first_name': 'Léon', 'last_name': 'Delacroix', 'birth_date': '1867-12-27', 'death_date': '1929-10-15', 'place_of_birth': 'Saint-Josse-ten-Noode', 'wikipedia_url': 'https://nl

### 1e. A sneak peak at the data (finally)

Look inside a few examples. Notice the dictionary keys available for each entry. You have your first example of *structured data*. This data was sanitized for your benefit, meaning it is readily exploitable without modification.

You will also notice there is a Wikipedia link for each entry. You will need to extract additional information there. This will be a case of *semi-structured* data.

The /countries endpoint returns a `list` of several country codes.

You need to loop through this list and query the /leaders endpoint for each one. Save each `json` result in a dictionary called `leaders_per_country`.

In [16]:
# Loop over the countries and store each leaders list (4 lines)
leaders_per_country = {}
for country in countries:
    leaders_per_country[country] = requests.get(
        leaders_url, cookies=cookies, params={"country": country}
    ).json()
print(leaders_per_country.keys())

dict_keys(['fr', 'us', 'be', 'ma', 'ru'])


In [17]:
# or 1 line (dictionary comprehension)
leaders_per_country = {country: requests.get(leaders_url, cookies=cookies, params={"country": country}).json() for country in countries}

It is finally time to create a `get_leaders()` function for the above code. You will build on it later-on. This function takes no parameter. Inside it, you will need to:
1. define the urls
2. get the cookies
2. get the countries
3. loop over them and save their leaders in a dictionary
4. return the dictionary

In [18]:
def get_leaders():
    # 1. define the urls
    root_url = "https://country-leaders.onrender.com"
    cookie_url = root_url + "/cookie"
    countries_url = root_url + "/countries"
    leaders_url = root_url + "/leaders"

    # 2. get the cookies
    cookies = requests.get(cookie_url).cookies

    # 3. get the countries
    countries = requests.get(countries_url, cookies=cookies).json()

    # 4. loop over them and save their leaders in a dictionary
    leaders_per_country = {}
    for country in countries:
        leaders_per_country[country] = requests.get(
            leaders_url, cookies=cookies, params={"country": country}
        ).json()

    # 5. return the dictionary
    return leaders_per_country

Test your function, save the result in the `leaders_per_country` dictionary and check its ouput.

In [19]:
leaders_per_country = get_leaders()
print(leaders_per_country.keys())

dict_keys(['fr', 'us', 'be', 'ma', 'ru'])


## 2. Extracting data from Wikipedia

Query one of the leaders' Wikipedia urls and display its `text` (not JSON).

In [20]:
leader = leaders_per_country["us"][0]
headers = {"User-Agent": "Mozilla/5.0 (Wikipedia scraper exercise)"}
print(requests.get(leader["wikipedia_url"], headers=headers).text[:1000])

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" lang="en" dir="ltr">
<head>
<meta charset="UTF-8">
<title>George Washington - Wikipedia</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled ve

Ouch! You get the raw HTML code of the webpage. If you try to deal with it without tools, you will be there all night. Instead, use the [beautiful soup 4](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) *external* library. You will find more info about it [here](../../2.python/2.python_advanced/05.Scraping/1.beautifulsoup_basic.ipynb) and [here](../../2.python/2.python_advanced/05.Scraping/2.beautifulsoup_advanced.ipynb)

Using the Quickstart section, start by importing the library and loading the output of your `get_text()` function.

Use the `prettify()` function and print it to take a look. You will start the actual parsing in the next step.

In [23]:
import sys
!{sys.executable} -m pip install beautifulsoup4

from bs4 import BeautifulSoup

soup = BeautifulSoup(requests.get(leader["wikipedia_url"], headers=headers).text, "html.parser")
print(soup.prettify()[:1000])

soup = BeautifulSoup(requests.get(leader["wikipedia_url"], headers=headers).text, "html.parser")
print(soup.prettify()[:1000])

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 107 kB 17.5 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard" dir="ltr" lang="en">
 <head>
  <meta charset="utf-8"/>
  <title>
   George Washington - Wikipedia
  </title>
  <script>
   (funct

That looks better but you need to extract the right part of the webpage: the text of the first paragraph.

It is a bit tricky because Wikipedia pages slightly differ in structure from one language to the next. We cannot simply get the text for the first HTML paragraph.

You will start by getting all the HTML paragraphs from the HTML source and saving them in the `paragraphs` variable.

Use the documentation or google the appropriate keywords.

In [24]:
# get all the HTML paragraphs from the source
paragraphs = soup.find_all("p")
print(len(paragraphs))

96


If you try different urls, you might find that the paragraph you want may be at a different index each time.

That is where you need to be clever and ask yourself what would be a reliable way to identify the right index ie. which string matches only the first paragraph whatever the language...

Spend a good 30 minutes on the problem and brainstorm with your fellow learners. If you come out empty handed, ask your coach.

1. Loop over the HTML paragraphs
2. When you have identified the correct one:
   * Store the [text](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#output) inside the `first_paragraph` variable
   * Exit the loop

In [25]:
first_paragraph = ""
for paragraph in paragraphs:
    if paragraph.find("b"):  # the lead paragraph bolds the subject's name in every language
        first_paragraph = paragraph.text
        break
print(first_paragraph)

George Washington (February 22, 1732 [O.S. February 11, 1731][a] – December 14, 1799) was a Founding Father and the first president of the United States, serving from 1789 to 1797. As commander of the Continental Army, Washington led Patriot forces to victory in the American Revolutionary War against the British Empire. He is commonly known as the Father of His Country for his role in bringing about American independence.



At this stage, you can create a function to maintain consistency in your code. We will give you its *skeleton*, you will copy the code you wrote and make it work inside a function.

Don't forget to test your function.

In [26]:
def get_first_paragraph(wikipedia_url):
    print(wikipedia_url)  # keep this for the rest of the notebook
    headers = {"User-Agent": "Mozilla/5.0 (Wikipedia scraper exercise)"}
    soup = BeautifulSoup(requests.get(wikipedia_url, headers=headers).text, "html.parser")
    first_paragraph = ""
    for paragraph in soup.find_all("p"):
        if paragraph.find("b"):
            first_paragraph = paragraph.text
            break
    return first_paragraph

In [27]:
# Test
url = leaders_per_country["fr"][0]["wikipedia_url"]
print(get_first_paragraph(url))

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
François Hollande [fʁɑ̃swa ɔlɑ̃d][n 3] Écouterⓘ, né le 12 août 1954 à Rouen (Seine-Inférieure), est un haut fonctionnaire et homme d'État français. Il est président de la République française du 15 mai 2012 au 14 mai 2017.



### 2a. Regular expressions to the rescue

Now that you have extracted the content of the first paragraph, the only thing that remains to finish your Wikipedia scraper is to sanitize the output.

Indeed some Wikipedia references, HTML code, phonetic pronunciation etc. may linger. You might find *regular expressions* handy to get rid of them and obtain pristine text. You will find some useful documentation about regular expressions [here](../../2.python/2.python_advanced/03.Regex/regex.ipynb)

Once you have one of your regex working online, try it in the cell below. 

Hints: 
* Check the `sub()` method documentation.
* Make sure to test urls in different languages. Some may look good but other do not.

In [28]:
import re

# remove the Wikipedia reference markers like [1], [a], [note 2]
clean = re.sub(r"\[[^\]]*\]", "", first_paragraph)
print(clean)

George Washington (February 22, 1732  – December 14, 1799) was a Founding Father and the first president of the United States, serving from 1789 to 1797. As commander of the Continental Army, Washington led Patriot forces to victory in the American Revolutionary War against the British Empire. He is commonly known as the Father of His Country for his role in bringing about American independence.



Overwrite the `get_first_paragraph()` function by applying your regex to the first paragraph before returning it.

In [30]:
def get_first_paragraph(wikipedia_url):
    print(wikipedia_url)  # keep this for the rest of the notebook
    headers = {"User-Agent": "Mozilla/5.0 (Wikipedia scraper exercise)"}
    soup = BeautifulSoup(requests.get(wikipedia_url, headers=headers).text, "html.parser")

    first_paragraph = ""
    for paragraph in soup.find_all("p"):
        if paragraph.find("b"):
            first_paragraph = paragraph.text
            break

    # remove reference markers like [1], [a], [note 2]
    first_paragraph = re.sub(r"\[[^\]]*\]", "", first_paragraph)
    return first_paragraph

Come up with other regexes to capture other patterns and sanitize the outputs completely. Modify your `get_first_paragraph()` function accordingly.

In [32]:
def get_first_paragraph(wikipedia_url):
    print(wikipedia_url)  # keep this for the rest of the notebook
    headers = {"User-Agent": "Mozilla/5.0 (Wikipedia scraper exercise)"}
    soup = BeautifulSoup(requests.get(wikipedia_url, headers=headers).text, "html.parser")

    first_paragraph = ""
    for paragraph in soup.find_all("p"):
        if paragraph.find("b"):
            first_paragraph = paragraph.text
            break

    # sanitize the text with a series of regular expressions
    first_paragraph = re.sub(r"\[[^\]]*\]", "", first_paragraph)        # [1], [a], [note 2]
    first_paragraph = re.sub(r"\([^()]*[\u24D8/][^()]*\)", "", first_paragraph)  # (listen ⓘ) / IPA ( /.../ )
    first_paragraph = re.sub(r"\s*\u24D8", "", first_paragraph)          # stray listen icons
    first_paragraph = re.sub(r"\s*\u00c9couter\b", "", first_paragraph)     # bare "listen" audio labels
    first_paragraph = first_paragraph.replace("\xa0", " ")                # non-breaking spaces
    first_paragraph = re.sub(r"\s+([,.;:])", r"\1", first_paragraph)      # space before punctuation
    first_paragraph = re.sub(r"\s{2,}", " ", first_paragraph).strip()     # collapse whitespace
    return first_paragraph

## 3. Putting it all together

Let's go back to your `get_leaders()` function and update it with an *inner* loop over each leader. You will query the url provided and extract the first paragraph using the `get_first_paragraph()` function you just finished. You will then update that `leader`'s dictionary and move on to the next one.

Notice, the rest of the code should not change since you modify the leader's data one by one.

In [33]:
def get_leaders():
    root_url = "https://country-leaders.onrender.com"
    cookie_url = root_url + "/cookie"
    countries_url = root_url + "/countries"
    leaders_url = root_url + "/leaders"

    cookies = requests.get(cookie_url).cookies
    countries = requests.get(countries_url, cookies=cookies).json()

    leaders_per_country = {}
    for country in countries:
        leaders = requests.get(leaders_url, cookies=cookies, params={"country": country}).json()
        # inner loop: enrich every leader with the first paragraph of their wiki page
        for leader in leaders:
            leader["first_paragraph"] = get_first_paragraph(leader["wikipedia_url"])
        leaders_per_country[country] = leaders

    return leaders_per_country

In [35]:
def get_leaders():
    root_url = "https://country-leaders.onrender.com"
    cookie_url = root_url + "/cookie"
    countries_url = root_url + "/countries"
    leaders_url = root_url + "/leaders"

    cookies = requests.get(cookie_url).cookies
    countries = requests.get(countries_url, cookies=cookies).json()

    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 (Wikipedia scraper exercise)"})

    leaders_per_country = {}
    for country in countries:
        req = requests.get(leaders_url, cookies=cookies, params={"country": country})
        try:
            leaders = req.json()
        except ValueError:
            leaders = None

        if req.status_code != 200 or not isinstance(leaders, list):
            cookies = requests.get(cookie_url).cookies
            req = requests.get(leaders_url, cookies=cookies, params={"country": country})
            req.raise_for_status()
            leaders = req.json()

            if not isinstance(leaders, list):
                raise RuntimeError(
                    f"Unexpected response from /leaders for {country}: {leaders!r}"
                )

        for leader in leaders:
            leader["first_paragraph"] = get_first_paragraph(
                leader["wikipedia_url"], session
            )
        leaders_per_country[country] = leaders

    return leaders_per_country
print(leaders_per_country["be"][0])

{'id': 'Q12978', 'first_name': 'Guy', 'last_name': 'Verhofstadt', 'birth_date': '1953-04-11', 'death_date': None, 'place_of_birth': 'Dendermonde', 'wikipedia_url': 'https://nl.wikipedia.org/wiki/Guy_Verhofstadt', 'start_mandate': '1999-07-12', 'end_mandate': '2008-03-20'}


Does the function crash in the middle of the loop? Chances are the cookies have expired while looping over the leaders.

Modify your function with an *exception* or check if the `status_code` is a cookie error. In either case, get new cookies and query the api again.

If your code did not crash,

In [36]:
def get_leaders():
    root_url = "https://country-leaders.onrender.com"
    cookie_url = root_url + "/cookie"
    countries_url = root_url + "/countries"
    leaders_url = root_url + "/leaders"

    cookies = requests.get(cookie_url).cookies
    countries = requests.get(countries_url, cookies=cookies).json()

    leaders_per_country = {}
    for country in countries:
        req = requests.get(leaders_url, cookies=cookies, params={"country": country})
        # the cookie may expire mid-loop: refresh it and retry
        if req.status_code != 200:
            cookies = requests.get(cookie_url).cookies
            req = requests.get(leaders_url, cookies=cookies, params={"country": country})
        leaders = req.json()

        for leader in leaders:
            leader["first_paragraph"] = get_first_paragraph(leader["wikipedia_url"])
        leaders_per_country[country] = leaders

    return leaders_per_country

Check the output of your function again.

In [37]:
leaders_per_country = get_leaders()
print(leaders_per_country["be"][0]["first_paragraph"])

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Val%C3%A9ry_Giscard_d%27Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers
https://fr.wikipedia.org/wiki/Napol%C3%A9on_III
https://fr.wikipedia.org/wiki/Paul_Doumer
https://fr.wikipedia.org/wiki/Alain_Poher
https://fr.wikipedia.org/wiki/Albert_Lebrun
https://fr.wikipedia.org/wiki/Ren%C3%A9_Coty
https://fr.wikipedia.org/wiki/Vincent_Auriol
https://fr.wikipedia.org/wiki/Patrice_de_Mac_Mahon
https://fr.wikipedia.org/wiki/%C3%89mile_Loubet
https://fr.wikipedia.org/wiki/Raymond_Poincar%C3%A9
https://fr.wikipedia.org/wiki/Sadi_Carnot_(homme_d%27%C3%89tat)
https://fr.wikipedia.org/wiki/Alexandre_Millerand
https://fr.wikipedia.org/wiki/Gaston_Doumergue
https://fr.wikipedia.

Well done! It took a while however... Let's speed things up. The main *bottleneck* is the loop. We call on the Wikipedia website many times.

You will use the same *session* to call all the wikipedia pages. Check the *Advanced Usage* section of the Requests module's documentation.

Start by modifying the `get_first_paragraph()` function to accept a session parameter and adjust the `get()` method call.

In [38]:
def get_first_paragraph(wikipedia_url, session):
    print(wikipedia_url)  # keep this for the rest of the notebook
    soup = BeautifulSoup(session.get(wikipedia_url).text, "html.parser")

    first_paragraph = ""
    for paragraph in soup.find_all("p"):
        if paragraph.find("b"):
            first_paragraph = paragraph.text
            break

    first_paragraph = re.sub(r"\[[^\]]*\]", "", first_paragraph)
    first_paragraph = re.sub(r"\([^()]*[\u24D8/][^()]*\)", "", first_paragraph)
    first_paragraph = re.sub(r"\s*\u24D8", "", first_paragraph)
    first_paragraph = re.sub(r"\s*\u00c9couter\b", "", first_paragraph)
    first_paragraph = first_paragraph.replace("\xa0", " ")
    first_paragraph = re.sub(r"\s+([,.;:])", r"\1", first_paragraph)
    first_paragraph = re.sub(r"\s{2,}", " ", first_paragraph).strip()
    return first_paragraph

Modify your `get_leaders()` function to make use of a single session for all the Wikipedia calls.
1. create a `Session` object outside of the loop over countries.
2. pass it to the `get_first_paragraph()` function as an argument.

In [39]:
def get_leaders():
    root_url = "https://country-leaders.onrender.com"
    cookie_url = root_url + "/cookie"
    countries_url = root_url + "/countries"
    leaders_url = root_url + "/leaders"

    cookies = requests.get(cookie_url).cookies
    countries = requests.get(countries_url, cookies=cookies).json()

    # 1. one Session reused for every Wikipedia call
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 (Wikipedia scraper exercise)"})

    leaders_per_country = {}
    for country in countries:
        req = requests.get(leaders_url, cookies=cookies, params={"country": country})
        if req.status_code != 200:  # refresh expired cookie and retry
            cookies = requests.get(cookie_url).cookies
            req = requests.get(leaders_url, cookies=cookies, params={"country": country})
        leaders = req.json()

        for leader in leaders:
            # 2. pass the shared session to get_first_paragraph()
            leader["first_paragraph"] = get_first_paragraph(leader["wikipedia_url"], session)
        leaders_per_country[country] = leaders

    return leaders_per_country

Test your new functions.



In [40]:
leaders_per_country = get_leaders()
print(leaders_per_country["fr"][0]["first_paragraph"])

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Val%C3%A9ry_Giscard_d%27Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers
https://fr.wikipedia.org/wiki/Napol%C3%A9on_III
https://fr.wikipedia.org/wiki/Paul_Doumer
https://fr.wikipedia.org/wiki/Alain_Poher
https://fr.wikipedia.org/wiki/Albert_Lebrun
https://fr.wikipedia.org/wiki/Ren%C3%A9_Coty
https://fr.wikipedia.org/wiki/Vincent_Auriol
https://fr.wikipedia.org/wiki/Patrice_de_Mac_Mahon
https://fr.wikipedia.org/wiki/%C3%89mile_Loubet
https://fr.wikipedia.org/wiki/Raymond_Poincar%C3%A9
https://fr.wikipedia.org/wiki/Sadi_Carnot_(homme_d%27%C3%89tat)
https://fr.wikipedia.org/wiki/Alexandre_Millerand
https://fr.wikipedia.org/wiki/Gaston_Doumergue
https://fr.wikipedia.

## 4. Saving your hard work

The final step is to save the ``leaders_per_country`` dictionary in the `leaders.json` file using the [json](https://docs.python.org/3/library/json.html) module. Check out the `with` statement.

In [41]:
import json

with open("leaders.json", "w", encoding="utf-8") as f:
    json.dump(leaders_per_country, f, ensure_ascii=False, indent=2)

Make sure the file can be read back. Write the code to read the file. And check the variables are the same.

In [42]:
with open("leaders.json", "r", encoding="utf-8") as f:
    reloaded = json.load(f)

print(reloaded == leaders_per_country)

True


Make a function `save(leaders_per_country)` to call this code easily.

In [43]:
def save(leaders_per_country):
    with open("leaders.json", "w", encoding="utf-8") as f:
        json.dump(leaders_per_country, f, ensure_ascii=False, indent=2)

In [44]:
save(leaders_per_country)

## 5. Tidy things up in a stand-alone python script

Congratulations! You now have a working scraper! However, your code is scattered throughout this notebook along side the tutorials. Hardly production ready...

Copy and paste what you need in a separate `leaders_scraper.py` file.
Make sure it works by calling `python3 leaders_scraper.py`

## (Optional) To go further

If you want to practice scraping, you can read this section and tackle the exercises.

1. Restructure your code by using OOP (see ReadMe).
2. You have noticed the API returns very partial results for country leaders. Many are missing. Overwrite the `get_leaders()` function to get its list from Wikipedia and extract their *personal details* from the frame on the side.

Good luck!